# Week 2: Dataset & Baseline: Group 3 Jupyter Notebook Lab Evaluator
**Team:** J013, J015, J016, J017, J018 · NLP Course Project 2026

> ⚠️ **SYNTHETIC DEVELOPMENT RUN.** The class notebooks, reference solutions and faculty-approved dataset
> are not yet available. This report runs the complete Week 2 pipeline on **team-authored synthetic fixtures
> with simulated annotations**. It demonstrates that the infrastructure works end to end. **None of the
> numbers below are evidence of how the evaluator performs on real student submissions.**

In [1]:
import json, sys, warnings
from pathlib import Path
import pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.pipeline import Pipeline
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
run = Pipeline(ROOT / "configs" / "synthetic_dev.yaml").run()
OUT = run.out
print("\ndata_status =", run.status)

=== data_status: SYNTHETIC_DEVELOPMENT_DATA ===
[anonymise] 10 notebooks, residual PII pattern matches after scrub: 0


[parse] 130 cell records (40 code, 90 markdown), 71 template cells flagged
[labels] 10 labelled notebooks, 2 flagged for discussion
[split] StratifiedGroupKFold(k=5, seed=42), strata=assignment/source; realised {'train': 0.6, 'val': 0.2, 'test': 0.2} (target {'train': 0.6, 'val': 0.2, 'test': 0.2})
[eval:week2_tfidf:code:val] n=2 ridge MAE=0.93 | mean-predictor MAE=2.50
[eval:week2_tfidf:code:test] n=2 ridge MAE=0.76 | mean-predictor MAE=2.00
[eval:week2_tfidf:explanation:val] n=2 ridge MAE=1.29 | mean-predictor MAE=2.50
[eval:week2_tfidf:explanation:test] n=2 ridge MAE=0.96 | mean-predictor MAE=1.00
[eval:week2_tfidf:documentation:val] n=2 ridge MAE=0.10 | mean-predictor MAE=0.00
[eval:week2_tfidf:documentation:test] n=2 ridge MAE=0.89 | mean-predictor MAE=1.50
[reports] wrote 10 per-notebook reports

data_status = SYNTHETIC_DEVELOPMENT_DATA


## 1. Project Overview
Programming labs are submitted as `.ipynb` files mixing code, markdown explanations, comments and outputs.
Auto-graders usually check only execution or output. Week 1 defined a component that evaluates **both** code
correctness and the natural-language content, returning a structured report (code correctness, output match,
explanation score, documentation quality, overall score, errors, feedback, confidence).

**Week 2 scope (assignment Section 14):** data collection, cleaning, preprocessing, annotation, split, baseline,
initial evaluation. Improved models (embeddings, CodeBERT, KeyBERT, LLM) are Weeks 3–4.

## 2. Dataset Description
| | Planned (Week 1) | Used in this run |
|---|---|---|
| Source | Anonymised class notebooks (MS Teams) + licensed GitHub/Kaggle notebooks | **Synthetic fixtures** authored by the team |
| Size | ~150–200 notebooks, ~3,000–4,000 cells | 10 notebooks, 2 labs × 5 quality profiles |
| Labels | ~100 notebooks × 3 annotators | **Simulated** marks from 3 fictitious annotators |

Synthetic labs: `LAB_SYN1` (TF-IDF & cosine similarity) and `LAB_SYN2` (bigram LM with add-one smoothing).
Notebooks were executed with a real kernel, so outputs, tracebacks and execution counts are genuine.
Each lab has a template, reference solution, reference explanations with concept lists, expected outputs
(derived by executing the reference) and a rubric. Public-notebook licences will be verified per source.

In [2]:
man = pd.read_csv(ROOT / "data/synthetic/manifest.csv")
man[["raw_path", "assignment_id", "group_key", "profile"]]

,raw_path,assignment_id,group_key,profile
0,raw/LAB_SYN1/SYN_S01_LAB_SYN1.ipynb,LAB_SYN1,SYN_S01,"excellent: correct code, docstrings/comments, ..."
1,raw/LAB_SYN1/SYN_S02_LAB_SYN1.ipynb,LAB_SYN1,SYN_S02,"correct code, no comments, weak/vague explanat..."
2,raw/LAB_SYN1/SYN_S03_LAB_SYN1.ipynb,LAB_SYN1,SYN_S03,"syntax error (T1), runtime NameError (T3), par..."
3,raw/LAB_SYN1/SYN_S04_LAB_SYN1.ipynb,LAB_SYN1,SYN_S04,"output mismatches (T1, T3), wrong construct (C..."
4,raw/LAB_SYN1/SYN_S05_LAB_SYN1.ipynb,LAB_SYN1,SYN_S05,"minimal: no explanations, no comments, T2 neve..."
5,raw/LAB_SYN2/SYN_S01_LAB_SYN2.ipynb,LAB_SYN2,SYN_S01,"excellent: correct code, docstrings/comments, ..."
6,raw/LAB_SYN2/SYN_S02_LAB_SYN2.ipynb,LAB_SYN2,SYN_S02,correct code with good comments/docstrings and...
7,raw/LAB_SYN2/SYN_S03_LAB_SYN2.ipynb,LAB_SYN2,SYN_S03,"runtime NameError (T2), syntax error (T3), mos..."
8,raw/LAB_SYN2/SYN_S04_LAB_SYN2.ipynb,LAB_SYN2,SYN_S04,"no smoothing (output mismatch), downstream Val..."
9,raw/LAB_SYN2/SYN_S05_LAB_SYN2.ipynb,LAB_SYN2,SYN_S05,"correct code, no comments, no explanations, co..."


## 3. Dataset Statistics

In [3]:
st = json.loads((OUT / "dataset_stats.json").read_text())
{k: v for k, v in st.items() if k != "label_distribution"}

{'data_status': 'SYNTHETIC_DEVELOPMENT_DATA',
 'n_notebooks': 10,
 'n_cells': 130,
 'cells_by_type': {'markdown': 90, 'code': 40},
 'template_cells': 71,
 'student_code_cells': 30,
 'student_markdown_cells': 19,
 'notebooks_by_assignment': {'LAB_SYN1': 5, 'LAB_SYN2': 5},
 'notebooks_by_source': {'synthetic': 10},
 'notebooks_by_split': {'train': 6, 'test': 2, 'val': 2},
 'cells_with_error_output': 5,
 'unexecuted_code_cells': 1,
 'pii_residual_matches': 0}

In [4]:
pd.DataFrame(st["label_distribution"]).fillna(0).astype(int).sort_index().rename_axis("mark")

,code,explanation,documentation
mark,,,
0.0,0,2,2
1.0,0,2,1
2.0,2,2,4
3.0,2,0,0
4.0,0,0,3
5.0,1,1,0
6.0,0,1,0
7.0,1,0,0
8.0,4,2,0


## 4. Notebook Parsing
`nbformat.read(as_version=4)` + `validate`. Each cell becomes a record with: type, source, execution count,
output types, text output (stream + `text/plain`, ANSI stripped), error name/value/traceback, an image flag
(bytes dropped), line/char counts. Cells are aligned to tasks via `## Task N` / `Q N` headers; unaligned
cells are left unassigned rather than guessed.

In [5]:
cells = pd.read_csv(OUT / "cells.csv")
ex = cells[cells.notebook_id == cells.notebook_id.iloc[26]]
ex[["cell_index", "cell_type", "task_id", "is_template", "execution_count", "has_error", "error_name", "output_text"]]

,cell_index,cell_type,task_id,is_template,execution_count,has_error,error_name,output_text
26,0,markdown,NaN,True,NaN,False,NaN,NaN
27,1,markdown,NaN,False,NaN,False,NaN,NaN
28,2,code,NaN,True,1.0,False,NaN,NaN
29,3,markdown,T1,True,NaN,False,NaN,NaN
30,4,code,T1,False,2.0,True,SyntaxError,NaN
31,5,markdown,T2,True,NaN,False,NaN,NaN
32,6,code,T2,False,3.0,False,NaN,"TF-IDF matrix shape: (3, 12)"
33,7,markdown,T2,True,NaN,False,NaN,NaN
34,8,markdown,T2,False,NaN,False,NaN,NaN
35,9,markdown,T3,True,NaN,False,NaN,NaN


## 5. Data Cleaning
* **Anonymisation first** (no external APIs): roll-number regex, e-mails, class-list names, user names in paths,
  all notebook/cell metadata. Files renamed to salted hashes. Two synthetic notebooks contain planted fake PII.
* **Template removal:** cells matching the handout (similarity ≥ 0.9) are flagged and excluded from student
  features, so prompt text cannot inflate similarity to the reference.
* **Placeholders** (`-`, "Your answer here", empty) are treated as no answer.

In [6]:
print("Residual PII pattern matches after anonymisation:", run.pii_residual)
raw = (ROOT / "data/synthetic/raw/LAB_SYN1/SYN_S05_LAB_SYN1.ipynb")
nid = run.manifest.set_index("raw_path").loc["raw/LAB_SYN1/SYN_S05_LAB_SYN1.ipynb", "notebook_id"]
import nbformat
print("BEFORE:", nbformat.read(raw, 4).cells[1].source.replace(chr(10), " "))
print("AFTER: ", nbformat.read(run.interim / f"{nid}.ipynb", 4).cells[1].source.replace(chr(10), " "))

Residual PII pattern matches after anonymisation: 0
BEFORE: **Name:** Fake Student Alpha  **Roll No:** J099  Contact: fake.student.alpha@example.com
AFTER:  **Name:** [NAME]  **Roll No:** [ROLL]  Contact: [EMAIL]


## 6. Preprocessing
**Code:** IPython magics/shell lines commented out → `ast.parse` (syntax check) → AST summary (calls, defs,
docstrings, loops, imports) → comments via `tokenize` (so `#` inside strings is ignored) → error type from
outputs, split into *environment* vs *code* errors → output normalisation (whitespace, case, memory addresses)
and comparison with expected outputs (numeric tolerance / contains / exact).

**Markdown:** images, links, HTML, header/emphasis marks removed; inline code and LaTeX content kept;
lowercased; token pattern keeps `tf-idf`, `word2vec`, `bi-lstm`; **no stemming or lemmatisation** so
technical terms survive; concept matching on token sequences with listed variants.

In [7]:
from src.preprocessing.text_clean import markdown_to_text, tokenize
from src.preprocessing import code_checks as cc
s = "## Why?\n**TF-IDF** down-weights `the`; see [docs](http://x) $P(w|c)$ and bi-LSTM"
print(markdown_to_text(s)); print(tokenize(markdown_to_text(s)))
print(cc.syntax_check("%time x = 1\nfor i in range(3)\n    print(i)"))
print(cc.extract_comments('s = "a # b"  # real comment'))

Why? TF-IDF down-weights the; see docs P(w|c) and bi-LSTM
['why', 'tf-idf', 'down-weights', 'the', 'see', 'docs', 'p', 'w', 'c', 'and', 'bi-lstm']
(False, "SyntaxError line 2: expected ':'")
['real comment']


## 7. Annotation Schema
Scales from the Week 1 example rubric: **Code 0–8, Explanation 0–8, Documentation 0–4**, marked per notebook.
Level descriptors are a **draft** (`docs/annotation_guidelines.md`) that the team must approve; Week 1 does not
define them. Procedure: joint calibration on ~10 notebooks → independent blind marking by 3 annotators →
items with annotator range > 1 discussed and adjudicated, others take the median. Agreement is reported on
pre-discussion marks. The mean pairwise annotator MAE estimates **human annotation variability**; it is a
reference point, not a hard ceiling. Below, agreement is computed on **simulated** annotations to test the code.

In [8]:
iaa = json.loads((OUT / "iaa.json").read_text())
pd.DataFrame({d: {k: r[k] for k in ["krippendorff_alpha_ordinal", "icc_2_1", "mean_pairwise_qwk",
                                     "mean_pairwise_mae", "mean_pairwise_within_1"]}
              for d, r in iaa["iaa"].items()}).round(3)

,code,explanation,documentation
krippendorff_alpha_ordinal,0.845,0.951,0.912
icc_2_1,0.940,0.971,0.908
mean_pairwise_qwk,0.934,0.968,0.898
mean_pairwise_mae,0.733,0.533,0.400
mean_pairwise_within_1,0.933,1.000,1.000


In [9]:
lab = pd.read_csv(OUT / "labels_final.csv")
lab[["notebook_id", "code", "explanation", "documentation", "code_needs_discussion",
     "explanation_needs_discussion", "documentation_needs_discussion"]]

,notebook_id,code,explanation,documentation,code_needs_discussion,explanation_needs_discussion,documentation_needs_discussion
0,01573e2e2521,8.0,8.0,4.0,False,False,False
1,08d1ce73b8d4,2.0,1.0,2.0,False,False,False
2,60cf8659878a,7.0,2.0,1.0,False,False,False
3,6a7732c244ad,5.0,0.0,0.0,True,False,False
4,6fc55a4498be,8.0,0.0,0.0,False,False,False
5,a290c5bd28bd,8.0,8.0,4.0,False,False,False
6,bced8ee14adb,8.0,2.0,4.0,False,False,False
7,be370b9034b8,2.0,1.0,2.0,False,False,False
8,c8a0c8c2e4e8,3.0,6.0,2.0,False,False,False
9,de5aaa0b5b63,3.0,5.0,2.0,True,False,False


## 8. Train/Validation/Test Split
Split at **notebook level**; all cells inherit their notebook's split. Groups are never divided: same student
(`group_key`), plus near-duplicate clusters (TF-IDF cosine ≥ 0.95, for copied/forked notebooks).
StratifiedGroupKFold (5 folds, stratified by assignment/source): 1 fold test, 1 validation, 3 train, targeting
**approximately** 60/20/20. Because whole groups are assigned, exact proportions are not guaranteed; realised
proportions are reported. TF-IDF vocabulary and imputation statistics are fit on train only.

*Why:* cells from one notebook share variable names, phrasing and style, and one student's labs share style.
Splitting cells randomly would let the model see near-copies of test material in training and inflate scores.

In [10]:
sp = json.loads((OUT / "split.json").read_text())
print(sp["method"]); print("target  ", sp["target"]); print("realised", sp["realised"], "| groups:", sp["n_groups"])
run.manifest.merge(pd.read_csv(ROOT/"data/synthetic/manifest.csv")[["raw_path"]], on="raw_path")\
   .groupby(["split", "group_key"]).assignment_id.apply(list)

StratifiedGroupKFold(k=5, seed=42), strata=assignment/source
target   {'train': 0.6, 'val': 0.2, 'test': 0.2}
realised {'train': 0.6, 'val': 0.2, 'test': 0.2} | groups: 5


split  group_key
test   SYN_S02      [LAB_SYN1, LAB_SYN2]
train  SYN_S01      [LAB_SYN1, LAB_SYN2]
       SYN_S04      [LAB_SYN1, LAB_SYN2]
       SYN_S05      [LAB_SYN1, LAB_SYN2]
val    SYN_S03      [LAB_SYN1, LAB_SYN2]
Name: assignment_id, dtype: object

## 9. Code-Based Baseline
Deterministic checks per notebook: share of student code cells that parse, share executed, error-cell rate,
out-of-order execution, output-match rate vs expected outputs, and required-construct coverage (non-generic
library calls in the reference solution, e.g. `TfidfVectorizer`, `cosine_similarity`; helper names defined by
the reference are excluded because students may name helpers differently).

In [11]:
F = run.df.merge(run.manifest[["notebook_id", "raw_path"]], on="notebook_id")
F["nb"] = F.raw_path.str.extract(r"(S0\d_LAB_SYN\d)")[0]
from src.models.features import CODE_FEATURES, EXPLANATION_FEATURES, DOCUMENTATION_FEATURES
F.set_index("nb").sort_index()[CODE_FEATURES + ["code"]].round(2)

,frac_syntax_ok,frac_executed,error_cell_rate,out_of_order,output_match_rate,construct_coverage,code
nb,,,,,,,
S01_LAB_SYN1,1.00,1.00,0.00,0.0,1.00,1.00,8.0
S01_LAB_SYN2,1.00,1.00,0.00,0.0,1.00,1.00,8.0
S02_LAB_SYN1,1.00,1.00,0.00,0.0,1.00,1.00,7.0
S02_LAB_SYN2,1.00,1.00,0.00,0.0,1.00,1.00,8.0
S03_LAB_SYN1,0.67,1.00,0.67,0.0,0.33,1.00,3.0
S03_LAB_SYN2,0.67,1.00,0.67,0.0,0.33,0.50,3.0
S04_LAB_SYN1,1.00,1.00,0.00,0.0,0.33,0.25,2.0
S04_LAB_SYN2,1.00,1.00,0.33,0.0,0.33,1.00,2.0
S05_LAB_SYN1,1.00,0.67,0.00,1.0,0.67,1.00,5.0


## 10. Text-Based Baseline
TF-IDF (unigrams + bigrams, sublinear tf, English stop words, custom technical tokeniser) **fit on training
explanations + reference texts only**; cosine similarity between each task's student explanation and the
reference explanation (mean and max over tasks). Concept coverage = share of reference concepts (with
variants) found in the explanation.

In [12]:
F.set_index("nb").sort_index()[EXPLANATION_FEATURES + ["explanation"]].round(2)

,tfidf_cos_mean,tfidf_cos_max,expl_tokens_log,frac_tasks_explained,keyword_coverage,explanation
nb,,,,,,
S01_LAB_SYN1,0.46,0.49,4.84,1.0,1.00,8.0
S01_LAB_SYN2,0.60,0.62,4.53,1.0,1.00,8.0
S02_LAB_SYN1,0.11,0.17,3.33,1.0,0.00,2.0
S02_LAB_SYN2,0.14,0.22,2.94,1.0,0.00,2.0
S03_LAB_SYN1,0.36,0.43,3.71,1.0,0.40,5.0
S03_LAB_SYN2,0.37,0.38,3.47,1.0,0.42,6.0
S04_LAB_SYN1,0.22,0.24,3.97,1.0,0.00,1.0
S04_LAB_SYN2,0.13,0.24,3.37,1.0,0.00,1.0
S05_LAB_SYN1,0.00,0.00,0.00,0.0,0.00,0.0


## 11. Feature Engineering
Documentation features: comment density (comment lines / code lines), docstring rate (docstrings / defined
functions), markdown length (log tokens), markdown-to-code cell ratio. Feature groups per dimension are set
in `src/models/features.py` and can be overridden from the config.

In [13]:
F.set_index("nb").sort_index()[DOCUMENTATION_FEATURES + ["documentation"]].round(2)

,comment_density,docstring_rate,md_tokens_log,md_to_code_ratio,documentation
nb,,,,,
S01_LAB_SYN1,0.13,1.0,4.84,0.67,4.0
S01_LAB_SYN2,0.09,1.0,4.53,0.67,4.0
S02_LAB_SYN1,0.00,0.0,3.33,0.67,1.0
S02_LAB_SYN2,0.32,1.0,2.94,0.67,4.0
S03_LAB_SYN1,0.30,0.0,3.71,0.67,2.0
S03_LAB_SYN2,0.07,0.0,3.47,0.67,2.0
S04_LAB_SYN1,0.17,0.0,3.97,0.67,2.0
S04_LAB_SYN2,0.07,0.0,3.37,0.67,2.0
S05_LAB_SYN1,0.00,0.0,0.00,0.00,0.0


## 12. Baseline Model
One **Ridge (linear) regression per rubric dimension** on standardised, median-imputed features;
alpha ∈ {0.1, 1, 10} chosen on validation MAE; predictions clipped to [0, max]. Linear rather than logistic
because Week 1 frames scoring as regression against human marks, the marks are ordinal, and a 9-class
logistic model would be unstable on ~60 training notebooks. A **mean predictor** is the reference floor.

In [14]:
met = json.loads((OUT / "metrics.json").read_text())
pd.DataFrame({d: r["coefficients"] for d, r in met["results"].items()}).round(3)

,code,explanation,documentation
frac_syntax_ok,0.000,NaN,NaN
frac_executed,0.093,NaN,NaN
error_cell_rate,-0.460,NaN,NaN
out_of_order,-0.093,NaN,NaN
output_match_rate,0.803,NaN,NaN
construct_coverage,0.460,NaN,NaN
_intercept,5.500,3.000,2.000
tfidf_cos_mean,NaN,0.839,NaN
tfidf_cos_max,NaN,0.743,NaN
expl_tokens_log,NaN,0.258,NaN


## 13. Evaluation
Per dimension: MAE, RMSE, Pearson and Spearman correlation, agreement within ±1 and exact agreement
(predictions rounded), bootstrap 95% CI for MAE when n ≥ 10. Correlations are reported as undefined when
n < 3 or a series is constant. Precision/recall/F1 are used only where a binary gold label exists (none in this
synthetic run). Concept-extraction F1 belongs with the Week 4 KeyBERT comparison.

## 14. Initial Results (SYNTHETIC: pipeline check only)
> ⚠️ Validation and test each contain **2 synthetic notebooks** with **simulated** labels. These numbers only show
> that training, prediction and scoring run correctly. They say nothing about accuracy on real submissions.

In [15]:
rows = []
for d, r in met["results"].items():
    for s in ("val", "test"):
        if s in r:
            for m in ("ridge", "mean_predictor"):
                x = r[s][m]
                rows.append({"dimension": d, "split": s, "model": m, "n": x["n"], "MAE": x["MAE"], "RMSE": x["RMSE"],
                             "pearson_r": x["pearson_r"], "within_1": x["within_1"], "exact": x["exact"]})
print(met["WARNING"])
pd.DataFrame(rows).round(2)

Synthetic development run: results are not evidence of real student performance. Computed on SYNTHETIC development fixtures with SIMULATED labels: demonstrates that the pipeline runs end to end, NOT how the evaluator performs on real submissions.


,dimension,split,model,n,MAE,RMSE,pearson_r,within_1,exact
0,code,val,ridge,2,0.93,1.02,None,1.0,0.0
1,code,val,mean_predictor,2,2.50,2.50,None,0.0,0.0
2,code,test,ridge,2,0.76,0.91,None,1.0,0.5
3,code,test,mean_predictor,2,2.00,2.06,None,0.5,0.0
4,explanation,val,ridge,2,1.29,1.40,None,0.5,0.0
5,explanation,val,mean_predictor,2,2.50,2.55,None,0.0,0.0
6,explanation,test,ridge,2,0.96,0.96,None,1.0,0.0
7,explanation,test,mean_predictor,2,1.00,1.00,None,1.0,0.0
8,documentation,val,ridge,2,0.10,0.14,None,1.0,1.0
9,documentation,val,mean_predictor,2,0.00,0.00,None,1.0,1.0


In [16]:
rep = json.loads(next((OUT / "reports").glob("*.json")).read_text())
print(json.dumps({k: rep[k] for k in list(rep)[:14]}, indent=1)[:1500])

{
 "data_status": "SYNTHETIC_DEVELOPMENT_DATA",
 "notebook_id": "08d1ce73b8d4",
 "assignment_id": "LAB_SYN2",
 "split": "train",
 "scoring_variant": "week2_tfidf",
 "code_correctness": 0.464,
 "output_match": false,
 "output_match_rate": 0.3333333333333333,
 "explanation_score": 1.25,
 "documentation_quality": 1.98,
 "overall_score": 6.95,
 "max_score": 20,
 "errors": [
  "cell 10 (T3): ValueError [code error] math domain error",
  "output mismatch in T2",
  "output mismatch in T3"
 ],
 "missing_concepts": {
  "T2": [
   "zero probability",
   "unseen bigrams",
   "add one",
   "vocabulary size",
   "probability mass"
  ],
  "T3": [
   "inverse probability",
   "normalised per word",
   "lower is better",
   "surprise"
  ]
 }
}


## 15. Limitations
* **No real data yet.** All results come from 10 synthetic notebooks with simulated labels; they validate code paths only.
* Rubric level descriptors are a draft awaiting team approval.
* Output comparison assumes tasks are marked with `## Task N` headers and that expected outputs exist.
* Keyword coverage is lexical: paraphrases outside the variant lists are missed and a wrong explanation that uses
  the right words can still score (the Week 1 stretch goal about code–explanation inconsistency needs Week 3–4 models).
* TF-IDF similarity rewards vocabulary overlap, not correctness.
* Notebooks are not re-executed by default (stored outputs are used); re-execution needs a sandbox and lab data files.
* With ~100 annotated notebooks, validation/test sets of ~20 give wide confidence intervals.

## 16. Week 2 Conclusion
**Done (infrastructure):** anonymiser, nbformat parser, task alignment and template removal, code checks
(AST, errors, output comparison, construct coverage), markdown cleaning and concept coverage, TF-IDF similarity
fit on train only, grouped notebook-level split, Ridge baseline with a mean-predictor floor, metrics and
agreement statistics, per-notebook JSON reports, 29 passing tests, config-driven so real data replaces
synthetic data without code changes.

**Not done (blocked on data):** real data collection, real annotation and agreement, and the real baseline
evaluation. These run with `python -m src.pipeline configs/class_data.yaml` once `data/class/` is populated.